## Imports

In [2]:
import os
import numpy as np
import scipy.io as sio
from pathlib import Path
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
%matplotlib inline
from matplotlib.colors import ListedColormap
import seaborn as sns
from tabulate import tabulate

## Define Dataset and output paths

In [17]:
DATA_DIR = Path.cwd().parent / "datasets"
OUTPUT_DIR = Path.cwd() / "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

In [18]:
def plot_band_correlation(data, img_label, filename):
    """Inter-band Pearson correlation heatmap (sampled pixels)."""
    H, W, B = data.shape
    flat = data.reshape(-1, B)
    flat = flat[pixel_valid_mask(flat)]
    pix = fill_missing_with_band_means(sample_pixels(flat, SAMPLE_PIXELS))
    corr = np.corrcoef(pix.T)   # (B, B)

    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto',
                   interpolation='nearest')
    ax.set_xlabel("Band Index")
    ax.set_ylabel("Band Index")
    ax.set_title(f"{img_label} — Inter-Band Correlation Matrix", fontweight='bold')
    cbar = fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pearson Correlation")
    plt.show()
    save(fig, filename)

In [19]:
def save(fig, filename):
    fig.savefig(os.path.join(OUTPUT_DIR, filename))
    plt.close(fig)
    print(f"  ✓ Saved: {filename}")

In [20]:
import os
import sys
import glob
import io
import re
import warnings
import numpy as np
import tifffile
import matplotlib.gridspec as gridspec
import seaborn as sns
from scipy import stats

In [21]:
BASE_DIR   = Path.cwd()
DATA_DIR   = os.path.join(BASE_DIR, "datas")
OUTPUT_DIR = os.path.join(BASE_DIR, "outputs", "week1b")

In [22]:
def extract_scene_date(scene_name):
    """Return YYYYMMDD extracted from an EnMAP scene folder name."""
    match = SCENE_DATE_RE.search(scene_name)
    return match.group(1) if match else None

In [23]:
def approx_wavelengths(n_bands):
    """Return a rough wavelength array [nm] for n EnMAP bands."""
    # VNIR 0–91 → 420–1000 nm; SWIR 92–n-1 → 1000–2450 nm
    n_vnir = min(92, n_bands)
    n_swir = max(0, n_bands - n_vnir)
    wl_vnir = np.linspace(420, 1000, n_vnir)
    wl_swir = np.linspace(1005, 2450, n_swir) if n_swir > 0 else np.array([])
    return np.concatenate([wl_vnir, wl_swir])

In [24]:
SAMPLE_PIXELS = 5000   # pixels to subsample for heavy computations
RANDOM_SEED   = 42
SCENE_DATE_RE = re.compile(r'_(\d{8})T\d{6}Z_')

In [25]:
def pixel_valid_mask(flat_pixels):
    """Keep pixels that contain at least one finite spectral value."""
    return np.isfinite(flat_pixels).any(axis=1)

In [26]:
def sample_pixels(data_2d_or_3d, n, seed=RANDOM_SEED):
    """Randomly sample n pixel rows from a (N, B) or (H*W, B) array."""
    rng = np.random.default_rng(seed)
    if data_2d_or_3d.ndim == 3:
        H, W, B = data_2d_or_3d.shape
        flat = data_2d_or_3d.reshape(-1, B)
    else:
        flat = data_2d_or_3d
    idx = rng.choice(len(flat), min(n, len(flat)), replace=False)
    return flat[idx]

In [27]:
def load_enmap_tif(filepath):
    """
    Load an EnMAP SPECTRAL_IMAGE.TIF and return (H, W, B) float32 array.
    Handles both (B,H,W) and (H,W,B) layouts.
    EnMAP nodata fill = -32768 (int16 origin). Replace with NaN.
    """
    data = tifffile.imread(filepath).astype(np.float64)
    # EnMAP ships as (Bands, H, W)
    if data.ndim == 3 and data.shape[0] > 100 and data.shape[1] > 500:
        data = data.transpose(1, 2, 0)   # → (H, W, B)
    # EnMAP nodata: -32768 (int16 min) or any value <= -10000
    data[data <= -10000] = np.nan
    return data

In [28]:
def fill_missing_with_band_means(flat_pixels):
    """
    Replace NaNs/inf in sampled pixels with band means.
    Used only for algorithms such as correlation that require finite input.
    """
    if len(flat_pixels) == 0:
        return flat_pixels

    filled = flat_pixels.copy()
    band_means = np.nanmean(filled, axis=0)
    band_means = np.where(np.isfinite(band_means), band_means, 0.0)

    missing = ~np.isfinite(filled)
    if missing.any():
        rows, cols = np.where(missing)
        filled[rows, cols] = band_means[cols]

    return filled

In [29]:
def main():
    search_pat = os.path.join(DATA_DIR, "**", "*-SPECTRAL_IMAGE.TIF")
    tif_files = sorted(glob.glob(search_pat, recursive=True))
    
    if not tif_files:
        print(f"[!] No *-SPECTRAL_IMAGE.TIF files found in {DATA_DIR}")
        return
    
    print(f"Found {len(tif_files)} EnMAP images:\n")
    for fp in tif_files:
        size_mb = os.path.getsize(fp) / 1024**2
        print(f"  • {os.path.basename(os.path.dirname(fp))[:60]}  ({size_mb:.0f} MB)")
    
    # ── Load FIRST image for detailed single-scene analysis ─────────────────
    for i, first_fp in enumerate(tif_files):
        scene_name = os.path.basename(os.path.dirname(first_fp))
        # Extract date string
        date_str = extract_scene_date(scene_name)
        if date_str:
            img_label = f"EnMAP Scene {i+1} ({date_str[:4]}-{date_str[4:6]}-{date_str[6:]})"
        else:
            img_label = f"EnMAP Scene {i+1}"
        
        print(f"\n{'─'*70}")
        print(f"  DETAILED ANALYSIS: {img_label}")
        print(f"{'─'*70}")
        print("  Loading TIF...")
        data_raw = load_enmap_tif(first_fp)
        H, W, B = data_raw.shape
        print(f"  Shape: {H} × {W} × {B}  |  dtype: {data_raw.dtype}")
        print(f"  Memory: {data_raw.nbytes / 1024**3:.2f} GB")
    
        # Approximate wavelengths
        wavelengths = approx_wavelengths(B)
        
        # NaN stats
        n_nodata = (~np.isfinite(data_raw).any(axis=2)).sum()
        n_valid  = H * W - n_nodata
        print(f"  Valid pixels : {n_valid:,} / {H*W:,}  "
              f"({100*n_valid/(H*W):.1f}%)")
        
        # Summary stats table
        flat = data_raw.reshape(-1, B)
        valid_mask = pixel_valid_mask(flat)
        complete_mask = np.isfinite(flat).all(axis=1)
        flat_valid = flat[valid_mask]
        n_complete = int(complete_mask.sum())
        print(f"  Fully observed spectra : {n_complete:,} / {H*W:,}  "
              f"({100*n_complete/(H*W):.1f}%)")
        
        if len(flat_valid) == 0:
            print("  [!] WARNING: No valid (non-nodata) pixels found after masking.")
            print("      Check nodata filter in load_enmap_tif().")
            return
        
        sample = sample_pixels(flat_valid, SAMPLE_PIXELS)
        print(f"\n  Per-pixel statistics ({len(sample):,} sampled pixels):")
        print(f"    Min          : {np.nanmin(sample):.4f}")
        print(f"    Max          : {np.nanmax(sample):.4f}")
        print(f"    Mean         : {np.nanmean(sample):.4f}")
        print(f"    Std          : {np.nanstd(sample):.4f}")
        print(f"    Median       : {np.nanmedian(sample):.4f}")
        print(f"    Bands        : {B}")

        # reshape to (pixels, bands)
        pixels = data_raw.reshape(-1, data_raw.shape[-1])
        
        # condition 1: all NaN
        all_nan = np.all(np.isnan(pixels), axis=0)
        
        # condition 2: zero variance
        std = np.nanstd(pixels, axis=0)
        zero_std = std < 1e-6
        
        # combine
        bad_bands = np.where(all_nan | zero_std)[0]
        
        print("Bad bands:", bad_bands)
        print("Number of bad bands:", len(bad_bands))

        plot_band_correlation(
            data_raw, img_label,
            filename=f"enmap_band_correlation {i+1}.png"
        )

In [ ]:
main()

Found 7 EnMAP images:

  • ENMAP01-____L2A-DT0000044635_20230923T050553Z_016_V010502_20  (346 MB)
  • ENMAP01-____L2A-DT0000044635_20230923T050607Z_019_V010502_20  (348 MB)
  • ENMAP01-____L2A-DT0000051947_20231128T051615Z_009_V010502_20  (351 MB)
  • ENMAP01-____L2A-DT0000058175_20240117T051315Z_001_V010502_20  (346 MB)
  • ENMAP01-____L2A-DT0000058175_20240117T051328Z_004_V010502_20  (350 MB)
  • ENMAP01-____L2A-DT0000060162_20240205T050519Z_014_V010502_20  (356 MB)
  • ENMAP01-____L2A-DT0000060162_20240205T050524Z_015_V010502_20  (355 MB)

──────────────────────────────────────────────────────────────────────
  DETAILED ANALYSIS: EnMAP Scene 1 (2023-09-23)
──────────────────────────────────────────────────────────────────────
  Loading TIF...
  Shape: 1176 × 1209 × 224  |  dtype: float64
  Memory: 2.37 GB
  Valid pixels : 1,035,446 / 1,421,784  (72.8%)
  Fully observed spectra : 0 / 1,421,784  (0.0%)

  Per-pixel statistics (5,000 sampled pixels):
    Min          : -454.0000
    Ma

C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_nanfunctions_impl.py:2019: RuntimeWarning: Degrees of freedom <= 0 for slice.
  var = nanvar(a, axis=axis, dtype=dtype, out=out, ddof=ddof,


Bad bands: [130 131 132 133 134]
Number of bad bands: 5


C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14640\1023557585.py:10: RuntimeWarning: Mean of empty slice
  band_means = np.nanmean(filled, axis=0)
C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3045: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
C:\Users\ADMIN\AppData\Local\Programs\Python\Python312\Lib\site-packages\numpy\lib\_function_base_impl.py:3046: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]
C:\Users\ADMIN\AppData\Local\Temp\ipykernel_14640\986149072.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


  ✓ Saved: enmap_band_correlation 1.png

──────────────────────────────────────────────────────────────────────
  DETAILED ANALYSIS: EnMAP Scene 2 (2023-09-23)
──────────────────────────────────────────────────────────────────────
  Loading TIF...


## Plot style configuration

In [23]:
import numpy as np

# reshape to (pixels, bands)
pixels = data_raw.reshape(-1, data.shape[-1])

# condition 1: all NaN
all_nan = np.all(np.isnan(pixels), axis=0)

# condition 2: zero variance
std = np.nanstd(pixels, axis=0)
zero_std = std < 1e-6

# combine
bad_bands = np.where(all_nan | zero_std)[0]

print("Bad bands:", bad_bands)
print("Number of bad bands:", len(bad_bands))

NameError: name 'data_raw' is not defined

In [16]:
plt.rcParams.update({
    'font.family': 'serif',
    'font.size': 11,
    'axes.titlesize': 13,
    'axes.labelsize': 12,
    'xtick.labelsize': 10,
    'ytick.labelsize': 10,
    'legend.fontsize': 9,
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'savefig.bbox': 'tight',
    'savefig.pad_inches': 0.1,
})

## Dataset Metadata

In [17]:
INDIAN_PINES_CLASSES = {
    0: "Background",
    1: "Alfalfa",
    2: "Corn-notill",
    3: "Corn-mintill",
    4: "Corn",
    5: "Grass-pasture",
    6: "Grass-trees",
    7: "Grass-pasture-mowed",
    8: "Hay-windrowed",
    9: "Oats",
    10: "Soybean-notill",
    11: "Soybean-mintill",
    12: "Soybean-clean",
    13: "Wheat",
    14: "Woods",
    15: "Buildings-Grass-Trees-Drives",
    16: "Stone-Steel-Towers",
}

PAVIA_UNIVERSITY_CLASSES = {
    0: "Background",
    1: "Asphalt",
    2: "Meadows",
    3: "Gravel",
    4: "Trees",
    5: "Painted metal sheets",
    6: "Bare Soil",
    7: "Bitumen",
    8: "Self-Blocking Bricks",
    9: "Shadows",
}

## Color palettes (distinguishable, colorblind-friendly-ish)

In [18]:
IP_COLORS = [
    "#000000",  # 0  Background (black)
    "#e6194b",  # 1  Alfalfa
    "#3cb44b",  # 2  Corn-notill
    "#ffe119",  # 3  Corn-mintill
    "#4363d8",  # 4  Corn
    "#f58231",  # 5  Grass-pasture
    "#911eb4",  # 6  Grass-trees
    "#42d4f4",  # 7  Grass-pasture-mowed
    "#f032e6",  # 8  Hay-windrowed
    "#bfef45",  # 9  Oats
    "#fabebe",  # 10 Soybean-notill
    "#469990",  # 11 Soybean-mintill
    "#e6beff",  # 12 Soybean-clean
    "#9A6324",  # 13 Wheat
    "#800000",  # 14 Woods
    "#aaffc3",  # 15 BGT-Drives
    "#808000",  # 16 Stone-Steel-Towers
]

PU_COLORS = [
    "#000000",  # 0  Background
    "#e6194b",  # 1  Asphalt
    "#3cb44b",  # 2  Meadows
    "#ffe119",  # 3  Gravel
    "#4363d8",  # 4  Trees
    "#f58231",  # 5  Painted metal sheets
    "#911eb4",  # 6  Bare Soil
    "#42d4f4",  # 7  Bitumen
    "#f032e6",  # 8  Self-Blocking Bricks
    "#bfef45",  # 9  Shadows
]

## Data Loading

In [19]:
def load_dataset(name):
    """
    Load a hyperspectral dataset from .mat files.
    
    Parameters
    ----------
    name : str
        Either 'indian_pines' or 'pavia_university'
    
    Returns
    -------
    data : np.ndarray, shape (H, W, B)
        Hyperspectral image cube.
    gt : np.ndarray, shape (H, W)
        Ground truth label map (0 = background/unlabeled).
    """
    if name == "indian_pines":
        data_mat = sio.loadmat(os.path.join(DATA_DIR, "Indian_pines_corrected.mat"))
        gt_mat = sio.loadmat(os.path.join(DATA_DIR, "Indian_pines_gt.mat"))
        # Identify the data key (not starting with __)
        data_key = [k for k in data_mat.keys() if not k.startswith("__")][0]
        gt_key = [k for k in gt_mat.keys() if not k.startswith("__")][0]
        data = data_mat[data_key].astype(np.float64)
        gt = gt_mat[gt_key].astype(np.int32)
    elif name == "pavia_university":
        data_mat = sio.loadmat(os.path.join(DATA_DIR, "PaviaU.mat"))
        gt_mat = sio.loadmat(os.path.join(DATA_DIR, "PaviaU_gt.mat"))
        data_key = [k for k in data_mat.keys() if not k.startswith("__")][0]
        gt_key = [k for k in gt_mat.keys() if not k.startswith("__")][0]
        data = data_mat[data_key].astype(np.float64)
        gt = gt_mat[gt_key].astype(np.int32)
    else:
        raise ValueError(f"Unknown dataset: {name}")
    
    return data, gt

In [20]:
# ─── Load Datasets ───────────────────────────────────────────────────
print("Loading datasets...")
ip_data, ip_gt = load_dataset("indian_pines")
pu_data, pu_gt = load_dataset("pavia_university")
print("  ✓ Indian Pines loaded")
print("  ✓ Pavia University loaded")

Loading datasets...


FileNotFoundError: [Errno 2] No such file or directory: 'E:\\SEM_6\\Minor Project\\datas\\Indian_pines_corrected.mat'

## Dataset Metadata & Statistics

In [ ]:
def print_dataset_info(data, gt, name, class_names):
    """Print comprehensive metadata about a dataset."""
    H, W, B = data.shape
    num_classes = len(class_names) - 1  # excluding background
    labeled_pixels = np.count_nonzero(gt)
    total_pixels = H * W
    memory_mb = data.nbytes / (1024 ** 2)
    
    print("\n" + "=" * 70)
    print(f"  DATASET: {name.upper().replace('_', ' ')}")
    print("=" * 70)
    print(f"  Spatial dimensions  : {H} × {W} pixels")
    print(f"  Spectral bands      : {B}")
    print(f"  Data type           : {data.dtype}")
    print(f"  Value range         : [{data.min():.2f}, {data.max():.2f}]")
    print(f"  Memory              : {memory_mb:.1f} MB")
    print(f"  Total pixels        : {total_pixels:,}")
    print(f"  Labeled pixels      : {labeled_pixels:,} ({100*labeled_pixels/total_pixels:.1f}%)")
    print(f"  Background pixels   : {total_pixels - labeled_pixels:,} ({100*(total_pixels-labeled_pixels)/total_pixels:.1f}%)")
    print(f"  Number of classes   : {num_classes}")
    print("-" * 70)
    
    # Per-class distribution
    unique, counts = np.unique(gt, return_counts=True)
    table_rows = []
    for cls_id, count in zip(unique, counts):
        if cls_id == 0:
            continue
        pct = 100 * count / labeled_pixels
        table_rows.append([
            cls_id,
            class_names[cls_id],
            f"{count:,}",
            f"{pct:.2f}%"
        ])
    
    print(tabulate(
        table_rows,
        headers=["ID", "Class Name", "Pixels", "% of Labeled"],
        tablefmt="simple_outline"
    ))
    
    # Class imbalance ratio
    class_counts = counts[unique != 0]
    imbalance_ratio = class_counts.max() / class_counts.min()
    print(f"\n  Max/Min class imbalance ratio: {imbalance_ratio:.1f}x")
    print(f"  Largest class : {class_names[unique[np.argmax(counts[unique != 0]) + 1]]} ({class_counts.max():,})")
    print(f"  Smallest class: {class_names[unique[np.argmin(counts[unique != 0]) + 1]]} ({class_counts.min():,})")
    print("=" * 70 + "\n")
    
    return unique, counts


In [ ]:
# ─── Dataset Metadata ────────────────────────────────────────────────
print("\n" + "─" * 70)
print("  DATASET METADATA & STATISTICS")
print("─" * 70)
print_dataset_info(ip_data, ip_gt, "Indian Pines", INDIAN_PINES_CLASSES)
print_dataset_info(pu_data, pu_gt, "Pavia University", PAVIA_UNIVERSITY_CLASSES)

## Visualization Functions

In [ ]:
def plot_false_color_composite(data, bands, title, filename):
    """
    Create and save a false-color RGB composite from selected bands.
    
    Parameters
    ----------
    data : np.ndarray (H, W, B)
    bands : tuple of 3 ints (R, G, B band indices)
    title : str
    filename : str
    """
    rgb = data[:, :, list(bands)].copy()
    # Per-channel stretch to [0, 1] for display
    for i in range(3):
        ch = rgb[:, :, i]
        p2, p98 = np.percentile(ch, [2, 98])
        rgb[:, :, i] = np.clip((ch - p2) / (p98 - p2 + 1e-8), 0, 1)
    
    fig, ax = plt.subplots(figsize=(7, 7))
    ax.imshow(rgb)
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    ax.tick_params(direction='in')
    
    # Band annotation
    band_text = f"R: Band {bands[0]}  |  G: Band {bands[1]}  |  B: Band {bands[2]}"
    ax.text(0.5, -0.08, band_text, transform=ax.transAxes, ha='center',
            fontsize=9, style='italic', color='gray')
    
    plt.savefig(os.path.join(OUTPUT_DIR, filename))
    plt.show()
    plt.close()
    print(f"  ✓ Saved: {filename}")

In [ ]:
def plot_ground_truth_map(gt, class_names, colors, title, filename):
    """
    Create and save a ground truth label map with legend.
    """
    n_classes = len(class_names)
    cmap = ListedColormap(colors[:n_classes])
    
    fig, ax = plt.subplots(figsize=(8, 8))
    im = ax.imshow(gt, cmap=cmap, vmin=0, vmax=n_classes - 1, interpolation='nearest')
    ax.set_title(title, fontweight='bold')
    ax.set_xlabel("Column")
    ax.set_ylabel("Row")
    
    # Create legend patches (skip background for cleaner legend)
    patches = [
        mpatches.Patch(color=colors[i], label=f"{i}: {class_names[i]}")
        for i in range(1, n_classes)
    ]
    ax.legend(
        handles=patches, loc='upper left', bbox_to_anchor=(1.02, 1.0),
        borderaxespad=0, fontsize=8, frameon=True,
        fancybox=True, shadow=True, title="Classes", title_fontsize=9
    )
    
    plt.savefig(os.path.join(OUTPUT_DIR, filename))
    plt.show()
    plt.close()
    print(f"  ✓ Saved: {filename}")

In [ ]:
def plot_spectral_signatures(data, gt, class_names, colors, title, filename):
    """
    Plot mean ± std spectral signature for each class.
    """
    n_classes = len(class_names)
    bands = np.arange(data.shape[2])
    
    fig, ax = plt.subplots(figsize=(12, 6))
    
    for cls_id in range(1, n_classes):
        mask = gt == cls_id
        if mask.sum() == 0:
            continue
        pixels = data[mask]  # shape: (N_pixels, B)
        mean_sig = pixels.mean(axis=0)
        std_sig = pixels.std(axis=0)
        
        ax.plot(bands, mean_sig, color=colors[cls_id], label=class_names[cls_id],
                linewidth=1.2, alpha=0.9)
        ax.fill_between(bands, mean_sig - std_sig, mean_sig + std_sig,
                        color=colors[cls_id], alpha=0.08)
    
    ax.set_xlabel("Band Index")
    ax.set_ylabel("Reflectance (Digital Number)")
    ax.set_title(title, fontweight='bold')
    ax.legend(loc='upper left', bbox_to_anchor=(1.02, 1.0),
              fontsize=7, frameon=True, fancybox=True, ncol=1)
    ax.grid(True, alpha=0.3, linestyle='--')
    ax.set_xlim(0, len(bands) - 1)
    
    plt.savefig(os.path.join(OUTPUT_DIR, filename))
    plt.show()
    plt.close()
    print(f"  ✓ Saved: {filename}")

In [ ]:
def plot_class_distribution(gt, class_names, colors, title, filename):
    """
    Plot horizontal bar chart showing per-class pixel count.
    """
    unique, counts = np.unique(gt, return_counts=True)
    
    # Filter out background
    mask = unique != 0
    cls_ids = unique[mask]
    cls_counts = counts[mask]
    
    # Sort by count (ascending for horizontal bar)
    sort_idx = np.argsort(cls_counts)
    cls_ids = cls_ids[sort_idx]
    cls_counts = cls_counts[sort_idx]
    
    labels = [f"{cid}: {class_names[cid]}" for cid in cls_ids]
    bar_colors = [colors[cid] for cid in cls_ids]
    
    fig, ax = plt.subplots(figsize=(10, max(5, len(cls_ids) * 0.45)))
    bars = ax.barh(labels, cls_counts, color=bar_colors, edgecolor='#333', linewidth=0.5)
    
    # Add count annotations
    for bar, count in zip(bars, cls_counts):
        ax.text(bar.get_width() + max(cls_counts) * 0.01, bar.get_y() + bar.get_height() / 2,
                f"{count:,}", va='center', fontsize=8, color='#333')
    
    ax.set_xlabel("Number of Pixels")
    ax.set_title(title, fontweight='bold')
    ax.set_xlim(0, max(cls_counts) * 1.15)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    
    plt.savefig(os.path.join(OUTPUT_DIR, filename))
    plt.show()
    plt.close()
    print(f"  ✓ Saved: {filename}")

In [ ]:
def plot_band_correlation(data, title, filename, sample_size=5000):
    """
    Plot inter-band correlation heatmap using a random pixel subset.
    """
    H, W, B = data.shape
    pixels = data.reshape(-1, B)
    
    # Subsample for speed
    if len(pixels) > sample_size:
        rng = np.random.default_rng(42)
        idx = rng.choice(len(pixels), sample_size, replace=False)
        pixels = pixels[idx]
    
    corr = np.corrcoef(pixels.T)
    
    fig, ax = plt.subplots(figsize=(8, 7))
    im = ax.imshow(corr, cmap='RdBu_r', vmin=-1, vmax=1, aspect='auto')
    ax.set_xlabel("Band Index")
    ax.set_ylabel("Band Index")
    ax.set_title(title, fontweight='bold')
    
    cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
    cbar.set_label("Pearson Correlation")
    
    plt.savefig(os.path.join(OUTPUT_DIR, filename))
    plt.close()
    print(f"  ✓ Saved: {filename}")

In [ ]:
def plot_combined_overview(data, gt, class_names, colors, bands_rgb, dataset_label, filename):
    """
    Create a 2×2 overview figure combining key visualizations.
    """
    n_classes = len(class_names)
    cmap = ListedColormap(colors[:n_classes])
    
    fig, axes = plt.subplots(2, 2, figsize=(14, 13))
    fig.suptitle(f"{dataset_label} — Dataset Overview", fontsize=15, fontweight='bold', y=0.98)
    
    # (0,0) False-color composite
    rgb = data[:, :, list(bands_rgb)].copy()
    for i in range(3):
        ch = rgb[:, :, i]
        p2, p98 = np.percentile(ch, [2, 98])
        rgb[:, :, i] = np.clip((ch - p2) / (p98 - p2 + 1e-8), 0, 1)
    axes[0, 0].imshow(rgb)
    axes[0, 0].set_title(f"False-Color Composite (B{bands_rgb[0]}, B{bands_rgb[1]}, B{bands_rgb[2]})")
    axes[0, 0].set_xlabel("Column")
    axes[0, 0].set_ylabel("Row")
    
    # (0,1) Ground truth
    axes[0, 1].imshow(gt, cmap=cmap, vmin=0, vmax=n_classes - 1, interpolation='nearest')
    axes[0, 1].set_title("Ground Truth Map")
    axes[0, 1].set_xlabel("Column")
    axes[0, 1].set_ylabel("Row")
    
    # (1,0) Spectral signatures
    band_arr = np.arange(data.shape[2])
    for cls_id in range(1, n_classes):
        mask = gt == cls_id
        if mask.sum() == 0:
            continue
        mean_sig = data[mask].mean(axis=0)
        axes[1, 0].plot(band_arr, mean_sig, color=colors[cls_id],
                        label=class_names[cls_id], linewidth=1.0, alpha=0.85)
    axes[1, 0].set_xlabel("Band Index")
    axes[1, 0].set_ylabel("Reflectance (DN)")
    axes[1, 0].set_title("Mean Spectral Signatures")
    axes[1, 0].legend(fontsize=6, loc='upper right', ncol=2, framealpha=0.7)
    axes[1, 0].grid(True, alpha=0.3, linestyle='--')
    
    # (1,1) Class distribution
    unique, counts = np.unique(gt, return_counts=True)
    mask_bg = unique != 0
    cls_ids = unique[mask_bg]
    cls_counts = counts[mask_bg]
    sort_idx = np.argsort(cls_counts)[::-1]
    cls_ids = cls_ids[sort_idx]
    cls_counts = cls_counts[sort_idx]
    x_labels = [f"{cid}" for cid in cls_ids]
    bar_colors = [colors[cid] for cid in cls_ids]
    axes[1, 1].bar(x_labels, cls_counts, color=bar_colors, edgecolor='#333', linewidth=0.5)
    axes[1, 1].set_xlabel("Class ID")
    axes[1, 1].set_ylabel("Pixel Count")
    axes[1, 1].set_title("Class Distribution")
    axes[1, 1].tick_params(axis='x', rotation=45)
    axes[1, 1].grid(axis='y', alpha=0.3, linestyle='--')
    
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, filename))
    plt.show()
    plt.close()
    print(f"  ✓ Saved: {filename}")

## Indian Pines Visualizations

In [ ]:
# Indian Pines Visualizations
print("\n" + "─" * 70)
print("  GENERATING INDIAN PINES VISUALIZATIONS")
print("─" * 70)

# False-color: Near-IR, Red, Green → bands 29, 19, 9 (approx)
plot_false_color_composite(
    ip_data, bands=(29, 19, 9),
    title="Indian Pines — False-Color Composite (NIR–R–G)",
    filename="ip_false_color.png"
)

plot_ground_truth_map(
    ip_gt, INDIAN_PINES_CLASSES, IP_COLORS,
    title="Indian Pines — Ground Truth Map (16 Classes)",
    filename="ip_ground_truth.png"
)

plot_spectral_signatures(
    ip_data, ip_gt, INDIAN_PINES_CLASSES, IP_COLORS,
    title="Indian Pines — Mean Spectral Signatures (±1 Std. Dev.)",
    filename="ip_spectral_signatures.png"
)

plot_class_distribution(
    ip_gt, INDIAN_PINES_CLASSES, IP_COLORS,
    title="Indian Pines — Class Distribution",
    filename="ip_class_distribution.png"
)

plot_band_correlation(
    ip_data,
    title="Indian Pines — Inter-Band Correlation Matrix",
    filename="ip_band_correlation.png"
)

In [ ]:
plot_combined_overview(
    ip_data, ip_gt, INDIAN_PINES_CLASSES, IP_COLORS,
    bands_rgb=(29, 19, 9), dataset_label="Indian Pines",
    filename="ip_overview.png"
)

## Pavia University Visualizations

In [ ]:
# ─── Pavia University Visualizations ─────────────────────────────────
print("\n" + "─" * 70)
print("  GENERATING PAVIA UNIVERSITY VISUALIZATIONS")
print("─" * 70)

# False-color: bands 56, 33, 12
plot_false_color_composite(
    pu_data, bands=(56, 33, 12),
    title="Pavia University — False-Color Composite",
    filename="pu_false_color.png"
)

plot_ground_truth_map(
    pu_gt, PAVIA_UNIVERSITY_CLASSES, PU_COLORS,
    title="Pavia University — Ground Truth Map (9 Classes)",
    filename="pu_ground_truth.png"
)

plot_spectral_signatures(
    pu_data, pu_gt, PAVIA_UNIVERSITY_CLASSES, PU_COLORS,
    title="Pavia University — Mean Spectral Signatures (±1 Std. Dev.)",
    filename="pu_spectral_signatures.png"
)

plot_class_distribution(
    pu_gt, PAVIA_UNIVERSITY_CLASSES, PU_COLORS,
    title="Pavia University — Class Distribution",
    filename="pu_class_distribution.png"
)

plot_band_correlation(
    pu_data,
    title="Pavia University — Inter-Band Correlation Matrix",
    filename="pu_band_correlation.png"
)

In [ ]:
plot_combined_overview(
    pu_data, pu_gt, PAVIA_UNIVERSITY_CLASSES, PU_COLORS,
    bands_rgb=(56, 33, 12), dataset_label="Pavia University",
    filename="pu_overview.png"
)

## Summary

In [ ]:
# print("\n" + "═" * 70)
# print("  Week 1 Complete — All figures saved to:")
# print(f"    {OUTPUT_DIR}")

# saved_files = sorted(os.listdir(OUTPUT_DIR))
# print(f"\n  Generated {len(saved_files)} files:")
# for f in saved_files:
#     size_kb = os.path.getsize(os.path.join(OUTPUT_DIR, f)) / 1024
#     print(f"    • {f} ({size_kb:.0f} KB)")
# print("═" * 70)